# Grover Search

Outline:
1) Oracle and Diffusion
2) Circuit version
3) Number of iterations
4) Multiple Solutions
5) What's next?

In [ ]:
import pennylane as qp
import pennylane.numpy as np

In [ ]:
n_qubits=3
dev = qp.device("default.qubit", wires=n_qubits)

Let's build the following operators:

1) Oracle Operator:  $V_f=\text{Id}-2\ket{a}\bra{a}$ s.t.:
\begin{equation*}
    V_f\ket{x}=(-1)^{f(x)}\ket{x}\begin{cases}
        \ket{x} & \quad x\neq a, \\
        -\!\ket{a} & \quad x= a. \\
    \end{cases}
\end{equation*}

2) Diffusion Operator:
$$ \ket{\Psi} = H\ket{0}_n = \frac{1}{2^{n/2}}\sum_{x=0}^{2^n-1}\ket{x}_n$$
$$    W = 2\ket{\Psi}\bra{\Psi} - I $$

Now, we will just write the matrices and Pennylane's function `qp.QubitUnitary` will directly create a quantum operation that implements that

In [ ]:
def oracle_matrix(combo):
    #Return a matrix representing the Oracle Operator
    ##### TODO #####
    return 


def diffusion_matrix():
    #Return a matrix representing the Diffusion Operator
    ##### TODO #####
    return 

@qp.qnode(dev)
def grover(combo):
    #Hadamard Transform
    ##### TODO #####
    
    #Oracle Operator
    ##### TODO #####
    
    #Diffusion Operator
    ##### TODO #####
    
    return qp.state()

Define a function called `grover_iter(combo,k)` where:
 1) you set the number of measurements and get the counts.
 2) Repeat the sequence of oracle and diffusion operators `k` times

In [ ]:
##### TODO #####

In [ ]:
from utils import plot_hist
combo=[0,1,0]

plot_hist(grover_iter(combo,1))

In [ ]:
plot_hist(grover_iter(combo,2))

# Let's do it with circuits

Now we will implement the same circuit via specific quantum gates, rather than with abstract unitaries The phase kickback will help us again with an additional qubit.

In [ ]:
def grover_circuit(combo,k):
    n_bits=len(combo)
    query_register = list(range(n_bits))
    aux = [n_bits]
    all_wires = query_register + aux

    def oracle(combo):
        # Hint: use "qp.MultiControlledX"
        ##### TODO #####

    def diffusion():
        # Hint: use "qp.MultiControlledX"
        ##### TODO #####

    qp.X(wires=aux)
    
    for wire in all_wires:
        qp.Hadamard(wires=wire)
    for _ in range(k):
        oracle(combo)
        diffusion()
    return qp.counts(wires=query_register)

combo=[0,1,0]

grover_node=qp.QNode(grover_circuit,shots=100,device=qp.device("default.qubit", wires=len(combo)+1))

plot_hist(grover_node(combo,2))

In [ ]:
qp.draw_mpl(grover_circuit,style = "pennylane")(combo,1)

## How many iterations?




$$\sin\theta = \frac{1}{\sqrt{N}} \;\Rightarrow\; \theta \approx \frac{1}{\sqrt{N}} \quad (N \text{ large})$$

$$\text{After } k \text{ steps: } P_{\text{success}} = \sin^2\!\big((2k+1)\theta\big)$$

$$\text{Maximal when } (2k+1)\theta = \frac{\pi}{2} \;\Rightarrow\; k = \frac{\pi}{4\theta} - \frac{1}{2} \approx \frac{\pi}{4}\sqrt{N}$$

In [ ]:
def zero_count(counts, n_bits):
    key = "0" * n_bits
    if key not in counts:
        return 0
    return counts[key]

def first_local_max(values, start=1):
    """Step number at the first peak of `values`, where values[i] <-> step start+i.
    It finds the first point where the success rate stops going up and starts going down.
    """
    if len(values) < 2:
        return start
    if values[0] > values[1]:
        return start
    for i in range(1, len(values) - 1):
        if values[i] >= values[i - 1] and values[i] > values[i + 1]:
            return start + i

In [ ]:
n_list = range(3, 7)
opt_steps = []
N_SHOTS=10000

for n_bits in n_list:
    combo = [0] * n_bits 
    step_list = range(1, 10)  
    num_list=[]
    dev = qp.device("default.qubit", wires=n_bits + 1, shots=N_SHOTS)
    grover_node = qp.QNode(grover_circuit, dev) 

    num_list = [zero_count(grover_node(combo, s), n_bits) for s in step_list]
    
    opt_steps.append(first_local_max(num_list, start=step_list[0]))

print(f"Approx : {[round((np.pi/4)*np.sqrt(2**el)) for el in n_list]}")

print("The optimal number of Grover steps for qubits in", n_list, "is", opt_steps, ".")

# Multiple Solutions

We can encode more than one solution!

In [ ]:

def grover_circuit(combos,k):
    n_bits=len(combos[0])
    query_register = list(range(n_bits))
    aux = [n_bits]
    all_wires = query_register + aux

    ##### TODO #####

    return qp.counts(wires=query_register)


combo=[[0,1,0],[1,0,1]]

grover_node=qp.QNode(grover_circuit,shots=10000,device=qp.device("default.qubit", wires=len(combo[0])+1))

plot_hist(grover_node(combo,1))

# What about costs?

Can we run grover on nowadays quantum devices? How do we compile abstract instructions to hardware level? How to define how costly a circuit is?

We'll discuss about this next week!